<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 6 — Structure in Many Dimensions</div>
</div>

#### Standardising, principal components and the noise floor, and k-means clustering checked against the truth: stars, galaxies and quasars from the Sloan Digital Sky Survey.
***

## Where this fits

Lecture 7 showed how variables vary together (covariance and correlation) and how to find the directions along which data really vary (the SVD and principal component analysis). Today you use those tools on real data, and add the last step: **clustering**, finding groups without being told what they are.

The question: **can you tell stars, galaxies and quasars apart from their images alone?** Large sky surveys image hundreds of millions of objects, but can take spectra of only a small fraction, so they need to sort objects from images first. You have 1800 objects from the Sloan Digital Sky Survey (SDSS): for each, its brightness in five colour bands and its size on the sky, which come from the images, and its true class, which comes from its spectrum. You'll find the structure without the labels, then use the labels to check what you found.

0. look at the data, and its uncertainties;
1. why standardise? a quick demonstration on wine;
2. principal component analysis by hand: variance, loadings and the noise floor;
3. k-means clustering, checked against the true classes.

These are the tools for Exercise 2, Task 5. **Nothing here is assessed.**

## How to use this notebook today

The in-class assessment runs during this lab, so this notebook is designed to work through on your own, in the lab or afterwards, and to serve as a guide for Exercise 2, Task 5. **Core (about 45 minutes):** Sections 0–3, except the parts marked *Optional*. The core cells already contain most of the code: fill in each `____`, run it, and compare with the **Check**. Everything marked *Optional* can wait.

**New to some of the Python?** Start with `week6_python_toolkit.ipynb` (15 minutes): standardising, `np.linalg.svd`, scikit-learn's `PCA` and `KMeans`, and comparing labels with `pd.crosstab`. If the basics themselves are shaky, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises them one at a time; each section below names the ones it uses.

Each task has a **Check** with the number you should get, and some have a collapsed hint.

In [ ]:
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week6/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

COL = {"star": "#FD9029", "galaxy": "#1F5FA8", "qso": "#B5121B"}     # one colour per class, used throughout

## <span style="color:#B5121B">0. Look at the data (10 minutes)</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 9 (just enough pandas); Section 8 (plots that say something). Each section takes 10–15 minutes.

Each object has dust-corrected magnitudes `mag_u` … `mag_z` in five bands from the ultraviolet (u) to the near-infrared (z), each with its uncertainty `err_u` … `err_z`, and a size `size_r` (arcsec) with its uncertainty. A magnitude is a logarithmic brightness: **smaller is brighter**, and the *difference* between two bands, a **colour** such as $u - g$, measures the shape of the object's spectrum, independent of its distance.

**(a)** Make the four colours $u-g$, $g-r$, $r-i$, $i-z$ and the logarithm of the size, $\log_{10}$(`size_r`). Plot $g-r$ against $u-g$, and a histogram of the log size, coloured by class. Which quantity separates galaxies from the rest? Which separates quasars from stars?

**(b)** The uncertainty of a colour is $\sqrt{\sigma_1^2 + \sigma_2^2}$ (independent errors, Lecture 5). Find the median uncertainty of each colour and compare it with the spread (standard deviation) of that colour across all 1800 objects.

In [ ]:
sky = pd.read_csv(data_path("sdss_balanced.csv"))
print(sky.shape, sky["class"].value_counts().to_dict())
sky.head(3)

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

```python
for a, b in [("u", "g"), ("g", "r"), ("r", "i"), ("i", "z")]:
    sky[f"{a}_{b}"] = sky[f"mag_{a}"] - sky[f"mag_{b}"]
    sky[f"err_{a}_{b}"] = np.hypot(sky[f"err_{a}"], sky[f"err_{b}"])
```
`np.hypot(x, y)` is $\sqrt{x^2+y^2}$. For the plot, loop over the classes and draw each in its own colour from `COL`.

</details>

In [ ]:
for a, b in [("u", "g"), ("g", "r"), ("r", "i"), ("i", "z")]:
    sky[f"{a}_{b}"] = sky[f"mag_{a}"] - sky[f"mag_{____}"]                      # the colour, e.g. u - g
    sky[f"err_{a}_{b}"] = np.hypot(sky[f"err_{a}"], sky[f"err_{b}"])          # its uncertainty
sky["log_size"] = np.log10(sky["size_r"])
sky["err_log_size"] = sky["size_err"] / (sky["size_r"] * np.log(10))          # error propagation for log10

FEATURES = ["u_g", "g_r", "r_i", "i_z", "log_size"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for c, grp in sky.groupby("class"):
    ax[0].plot(grp["u_g"], grp["g_r"], ".", ms=3, color=COL[c], label=c)
    ax[1].hist(grp["log_size"], bins=40, histtype="step", color=COL[c], label=c)
ax[0].set(xlabel="u - g (mag)", ylabel="g - r (mag)", xlim=(-0.5, 3.5), ylim=(-0.5, 2)); ax[0].legend()
ax[1].set(xlabel="log10(size / arcsec)", ylabel="number"); ax[1].legend()
plt.tight_layout(); plt.show()

for f in FEATURES:
    print(f"{f:9s} median uncertainty {sky['err_' + f].median():.4f}   spread {sky[f].std():.3f}")

**Check:** median $u-g$: stars 1.21, galaxies 1.74, quasars 0.25; median size about 1.2 arcsec for stars and quasars (the blur of the atmosphere: they are points) and 4.9 arcsec for galaxies. The uncertainties are tiny compared with the spreads: 0.033 against 0.73 mag for $u-g$, 0.010 against 0.43 for $g-r$.

## <span style="color:#B5121B">1. Why standardise? A demonstration on wine (10 minutes)</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 1 (NumPy arrays); Section 6 (functions). Each section takes 10–15 minutes.

Before the stars, a smaller dataset where the point is impossible to miss: 13 chemical measurements of 178 Italian wines from three grape varieties (cultivars). The measurements are in very different units and sizes: proline is in the hundreds, hue is about 1.

**(a)** Carry out PCA on the raw measurements (subtract each column's mean, then the SVD, Lecture 7). What fraction of the variance does the first component carry, and which measurement dominates it?

**(b)** Repeat on **standardised** measurements: subtract each mean and divide by each standard deviation, so every column has mean 0 and standard deviation 1.

**(c)** Cluster the wines into 3 groups with k-means, raw and standardised. How many wines land in the group that matches their cultivar?

In [ ]:
wine = pd.read_csv(data_path("wine.csv"))
W = wine.drop(columns="cultivar").to_numpy()             # 178 wines x 13 measurements
cultivar = wine["cultivar"].to_numpy()
print(wine.drop(columns="cultivar").describe().loc[["mean", "std"]].round(2).T)

def pca(X):
    """PCA of the rows of X via the SVD: (fraction of variance per component, directions as rows)."""
    Xc = X - X.mean(axis=0)
    U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    return s**2 / np.sum(s**2), Vt

def best_match(labels, truth):
    """Largest number of objects whose cluster matches their true class, over every way of naming the clusters."""
    classes = np.unique(truth)
    return max(sum(np.sum((labels == p[i]) & (truth == c)) for i, c in enumerate(classes))
               for p in itertools.permutations(np.unique(labels), len(classes)))

In [ ]:
names = wine.columns[:-1]
W_std = (W - W.mean(axis=0)) / W.std(axis=0, ddof=1)                     # standardised: mean 0, sd 1
for label, X in [("raw", W), ("standardised", ____)]:
    frac, Vt = pca(X)
    top = names[np.argmax(np.abs(Vt[0]))]
    km = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)
    print(f"{label:13s} PC1 carries {frac[0]:.1%}, mostly {top}; k-means matches {best_match(km, cultivar)} of 178")

**Check:** raw: PC1 carries 99.8% of the variance and is almost pure proline, and k-means matches 125 of 178 wines (70%). Standardised: PC1 carries 36% (flavanoids, phenols and OD280 together), and k-means matches 172 of 178 (97%). With raw numbers, PCA and k-means both see only the measurement with the biggest numbers.

## <span style="color:#B5121B">2. Principal components of the sky (20 minutes)</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 1 (NumPy arrays); Section 3 (indexing and slicing); Section 8 (plots that say something). Each section takes 10–15 minutes.

Back to the 1800 objects, described by the five quantities in `FEATURES`. Colours are in magnitudes, the size is a logarithm: different quantities, so **standardise first**. Why the logarithm of the size? Sizes are positive and skewed, spanning a factor of about 100 from the smallest star to the largest galaxy; without the log, a few huge galaxies would dominate the variance. Take logs of positive, skewed quantities before PCA (in Exercise 2: the band powers).

**(a)** Standardise, take the SVD, and print the fraction of the variance carried by each of the five components. How many do you need for 90%?

**(b)** Print the **loadings** of the first two components (the rows of `Vt`) as a table with one row per feature. In words, what does each component measure? (The sign of a component is arbitrary.)

**(c)** Plot every object on the first two components, coloured by its true class. Which classes separate, and along which component? Now combine the plot with the loadings: **which measured quantities** drive that separation? (This is the reasoning Exercise 2, Task 5 asks for: from components back to the quantities you measured.)

**(d) Optional: the noise floor.** After standardising, the measurement noise in feature $j$ has variance about $\sigma_j^2 / s_j^2$, where $\sigma_j$ is its typical uncertainty and $s_j$ its spread. Add these up over the five features and compare with the variance of the smallest component (Lecture 7: a component whose variance is no bigger than the noise carries no information).

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

```python
X = sky[FEATURES].to_numpy()
mean, sd = X.mean(axis=0), X.std(axis=0, ddof=1)
Z = (X - mean) / sd
U, s, Vt = np.linalg.svd(Z, full_matrices=False)
var = s**2 / (len(Z) - 1)             # variance along each direction
scores = Z @ Vt.T                      # each object's coordinates on the components
```
Each row of `Vt` is one component; `pd.DataFrame(Vt[:2].T, index=FEATURES, columns=["PC1", "PC2"])` makes the loadings table.

</details>

In [ ]:
X = sky[FEATURES].to_numpy()
mean, sd = X.mean(axis=0), X.std(axis=0, ddof=1)
Z = (X - mean) / ____                                         # standardise
U, s, Vt = np.linalg.svd(Z, full_matrices=False)
var = s**2 / (len(Z) - 1)                                     # variance along each component
print("fraction of variance:", np.round(var / var.sum(), 3))
print(pd.DataFrame(Vt[:2].T, index=FEATURES, columns=["PC1", "PC2"]).round(2))

scores = Z @ Vt.T
for c, idx in sky.groupby("class").groups.items():
    plt.plot(scores[idx, 0], scores[idx, 1], ".", ms=3, color=COL[c], label=c)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend(); plt.show()

# (d) optional: the noise floor
noise = sum(sky["err_" + f].median()**2 / sd[j]**2 for j, f in enumerate(FEATURES))
print(f"noise variance about {noise:.3f}; smallest component {var[-1]:.3f}")

**Check:** 71%, 13.5%, 8%, 5.4% and 2.1%: three components carry 92%. PC1 loads all five features with the same sign (about 0.35–0.50): redder and bigger at one end, bluer and point-like at the other. PC2 is mostly size (about −0.87), with some $u-g$. Galaxies separate along PC1 and PC2; stars and quasars overlap but are offset. The noise variance is about 0.018, so even the smallest component (0.107) is well above the noise floor: SDSS photometry is precise.

## <span style="color:#B5121B">3. Clusters without labels (20 minutes)</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 5 (loops that fill an array); Section 9 (just enough pandas). Each section takes 10–15 minutes.

> **Primer: k-means.** Choose $k$. Start with $k$ centres, then repeat two steps: assign every point to its nearest centre; move each centre to the mean of its points. Stop when nothing changes. k-means looks for $k$ **round groups of similar size**, and the answer can depend on where it started, so run it from several starts (`n_init=10`) and keep the best. The cluster numbers it returns are arbitrary: cluster 0 is not "star".

**(a)** Cluster the standardised data `Z` into $k = 3$ groups. Make a table of counts: true class (rows) against cluster (columns), with `pd.crosstab`. Do the three clusters match the three classes? Which classes are mixed together?

**(b)** Try $k = 4$. Look at the galaxies: how are they split? Compare the median $u - r$ colour of the galaxies in each of their clusters.

**(c) Optional:** for each $k$ from 1 to 8, record k-means' *inertia* (the total squared distance of points from their centres). Plot it against $k$. Is there an obvious best $k$?

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

```python
labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(Z)
print(pd.crosstab(sky["class"], labels))
```
For (b), select the galaxies with a mask, `gal = sky["class"] == "galaxy"`, and group them by their cluster label. For (c), `KMeans(...).fit(Z).inertia_`.

</details>

In [ ]:
labels3 = KMeans(n_clusters=____, n_init=10, random_state=0).fit_predict(Z)
print(pd.crosstab(sky["class"], labels3))
print("best match:", best_match(labels3, sky["class"].to_numpy()), "of", len(sky))

sky["k4"] = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(Z)
print(pd.crosstab(sky["class"], sky["k4"]))
gal = sky["class"] == "galaxy"
print((sky.loc[gal, "mag_u"] - sky.loc[gal, "mag_r"]).groupby(sky.loc[gal, "k4"]).agg(["size", "median"]).round(2))

# (c) optional: inertia against k
inertia = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z).inertia_ for k in range(1, 9)]
plt.plot(range(1, 9), inertia, "o-"); plt.xlabel("k"); plt.ylabel("inertia"); plt.show()

**Check:** with $k = 3$ the clusters do **not** match the classes: the galaxies are split between two clusters (289 and 305), and most stars (472 of 600) share a cluster with the quasars; the best match is 965 of 1800. With $k = 4$ the galaxies are still split in two, but stars and quasars now get clusters of their own (441 and 480): named after their largest class, the clusters are right for 1514 of 1800. The two galaxy clusters have median $u-r$ of about 2.1 and 3.0, and 55% against 3% have star-forming spectra: the blue and the red galaxies. The inertia falls smoothly (8995, 3949, 3301, 2761, …): no sharp elbow.

## <span style="color:#FD9029">4. Optional: the real sky</span>

Today's sample has 600 of each class. The real sky doesn't: in `sdss_sky.csv` (all 10 947 objects that passed the same quality cuts), 58% are galaxies, 37% stars and only 6% quasars.

Repeat Section 3 on the full sample: standardise it, cluster with $k = 3$ and $k = 4$, and make the tables. Where do the quasars go? If you judged k-means by "the fraction of objects in a cluster named after their class", would you notice? (This is why, when we meet classification later in the course, accuracy alone will not be enough.)

In [ ]:
# Optional

## <span style="color:#FD9029">5. Optional: k-means by hand</span>

Write k-means yourself, following the primer: start from $k$ randomly chosen objects as centres, then alternate *assign* and *update* until no label changes. Run it on `Z` with $k = 4$ and compare its inertia with scikit-learn's. Run it from a few different starts: do you always get the same answer?

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

Distances from every point to every centre in one line: `d2 = ((Z[:, None, :] - centres[None, :, :])**2).sum(axis=2)`, an array of shape (number of points, k). Then `labels = d2.argmin(axis=1)`, and the new centre of cluster `j` is `Z[labels == j].mean(axis=0)`.

</details>

In [ ]:
# Optional

## <span style="color:#FD9029">6. Optional: what does size add?</span>

Drop `log_size` and cluster on the four colours alone ($k = 4$). How many objects are now in a cluster named after their class? Which class suffers most, and why? (Think about what the size measures, and what the colours of a galaxy at redshift 0.1 look like.)

In [ ]:
# Optional

## Summary

- **Look first,** and compare the uncertainties with the spread: here the measurements are far more precise than the differences between objects.
- **Standardise** quantities in different units before PCA or clustering, or the biggest numbers win (wine: 99.8% "proline").
- **PCA** finds the directions of largest variance. Read the **loadings** to say what each component means; compare small components with the **noise floor** before interpreting them.
- **k-means** finds $k$ round groups of similar size. Its clusters are not your classes: it can split a class with real internal structure (red and blue galaxies) and merge classes that look alike (hot stars and quasars). Check clusters against known labels with a **table of counts**, and remember that cluster numbers are arbitrary.
- **Rare classes** can vanish in the real, unbalanced data while the overall agreement still looks high.

For Exercise 2, Task 5: standardise, PCA with loadings, the people on the first two components coloured by group, then k-means with $k = 2$ compared with the groups in a table.

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>